# 01 · Preprocesamiento

Equivalente a `1_preprocess_dates.ipynb` + `2_limpiar_crear_chunks.ipynb` de Karen.

**Diferencia clave:** los tweets son cortos (~30 palabras), no articulos de miles de palabras.
Por eso no se aplica chunking solapado: **cada tweet = 1 chunk**.
El archivo `chunks.parquet` mantiene la misma estructura que el de Karen para que el resto del pipeline sea compatible.

In [ ]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path

# -- Rutas ---------------------------------------------------
ARCHIVO_TWEETS = Path(r'C:\Users\afpue\Documents\GitHub\icare\archivos\df_twitter.csv')
DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosComportamiento')

# Crear carpeta si no existe (pathlib.mkdir es confiable en Windows, sin subprocess)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print(f'[OK] DATA_PROCESSED : {DATA_PROCESSED}')
print(f'     Existe en disco : {DATA_PROCESSED.exists()}')

# -- Columnas del DataFrame ----------------------------------
COL_TEXTO       = 'Content'
COL_FECHA       = 'Fecha'
COL_DATE        = 'Date'
COL_AUTOR       = 'Author_Normalized'
COL_AUTOR_NAME  = 'Author Name'
COL_LOCATION    = 'Location'
COL_LIKES       = 'Number of Likes'
COL_RTS         = 'Number of Retweets'
COL_HASHTAGS    = 'Hashtags'
COL_MENTIONS    = 'Mentions'
COL_SENTIMIENTO = 'Sentimiento'
COL_POLARIDAD   = 'Polaridad'
COL_ENTIDAD     = 'Entidad'

print(f'  ARCHIVO_TWEETS : {ARCHIVO_TWEETS}')

In [ ]:
# ============================================================
# CELL 1 - CARGA
# Equivalente a: corpus_completo = pd.read_excel(ruta_corpus)
# ============================================================
import pandas as pd

df = pd.read_csv(ARCHIVO_TWEETS, low_memory=False)
print(f'[CARGA] Leido desde disco: {ARCHIVO_TWEETS}')
print(f'  Filas    : {len(df):,}')
print(f'  Columnas : {list(df.columns)}')
df.head(3)

In [ ]:
# ============================================================
# CELL 2 - FECHAS
# Equivalente a: corpus_completo = utils.aplicar_funcion_fecha(corpus_completo)
# ============================================================

df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors='coerce')
df[COL_DATE]  = pd.to_datetime(df[COL_DATE],  errors='coerce')

n_nulos = df[COL_FECHA].isna().sum()
print(f'[FECHAS] Tweets con fecha valida : {len(df) - n_nulos:,}')
print(f'[FECHAS] Tweets sin fecha        : {n_nulos:,}')
print(f'[FECHAS] Rango                   : {df[COL_FECHA].min()} -> {df[COL_FECHA].max()}')

df['anio']     = df[COL_FECHA].dt.year
df['mes']      = df[COL_FECHA].dt.month
# .astype(str) evita el tipo Period que pyarrow no serializa
df['anio_mes'] = df[COL_FECHA].dt.to_period('M').astype(str)

df[[COL_FECHA, COL_DATE, 'anio', 'mes', 'anio_mes']].head(3)

In [ ]:
# ============================================================
# CELL 3 - LIMPIEZA
# Equivalente a: corpus = utils.aplicar_funcion_limpieza(corpus)
# ============================================================
import re

def limpiar_tweet(texto: str) -> str:
    """Limpia un tweet: quita RT, URLs, menciones, caracteres especiales."""
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'^RT\s+',                  '',  texto, flags=re.IGNORECASE)
    texto = re.sub(r'http\S+|www\S+|https\S+', ' ', texto)
    texto = re.sub(r'@\w+',                    ' ', texto)
    texto = re.sub(r'[^A-Za-z\u00e0-\u00ff\s\.\,\;\:\!\?\u00bf\u00a10-9]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['Texto_limpio'] = df[COL_TEXTO].apply(limpiar_tweet)

n_vacios = (df['Texto_limpio'].str.strip() == '').sum()
print(f'[LIMPIEZA] Tweets vacios tras limpieza : {n_vacios:,}')

df = df[df['Texto_limpio'].str.strip() != ''].reset_index(drop=True)
print(f'[LIMPIEZA] Tweets conservados          : {len(df):,}')

df['n_palabras'] = df['Texto_limpio'].str.split().str.len()
print(f"[LIMPIEZA] Palabras/tweet  media: {df['n_palabras'].mean():.1f}  "
      f"mediana: {df['n_palabras'].median():.0f}  "
      f"max: {df['n_palabras'].max()}")

df[[COL_TEXTO, 'Texto_limpio', 'n_palabras']].head(3)

In [ ]:
# ============================================================
# CELL 4 - ID UNICO
# En utils.crear_chunks() del paper: id_doc = idx + 1
# ============================================================

df = df.reset_index(drop=True)
df['id_doc'] = df.index + 1

print(f"[ID] Rango id_doc : {df['id_doc'].min()} -> {df['id_doc'].max()}")
df[['id_doc', COL_AUTOR, COL_FECHA, 'Texto_limpio']].head(3)

In [ ]:
# ============================================================
# CELL 5 - GUARDAR CORPUS LIMPIO
# Equivalente a: corpus.to_excel(..., 'corpus_cleaned.xlsx') en Karen NB2
# ============================================================

COLS_CORPUS = [
    'id_doc',
    COL_AUTOR, COL_AUTOR_NAME, COL_FECHA, COL_DATE,
    'anio', 'mes', 'anio_mes',
    COL_LOCATION, COL_LIKES, COL_RTS,
    COL_HASHTAGS, COL_MENTIONS,
    COL_SENTIMIENTO, COL_POLARIDAD, COL_ENTIDAD,
    COL_TEXTO, 'Texto_limpio', 'n_palabras',
]
COLS_CORPUS = [c for c in COLS_CORPUS if c in df.columns]

corpus_limpio = df[COLS_CORPUS].copy()

ruta_parquet = DATA_PROCESSED / 'corpus_cleaned.parquet'
ruta_excel   = DATA_PROCESSED / 'corpus_cleaned.xlsx'

corpus_limpio.to_parquet(ruta_parquet, index=False, engine='pyarrow')
print(f'[GUARDADO] corpus_cleaned.parquet : {len(corpus_limpio):,} filas')

corpus_limpio.to_excel(ruta_excel, index=False, engine='openpyxl')
print(f'[GUARDADO] corpus_cleaned.xlsx    : {len(corpus_limpio):,} filas')
print(f'  Ruta: {DATA_PROCESSED}')

In [ ]:
# ============================================================
# CELL 6 - CREAR CHUNKS
# Equivalente a: chunks_df = utils.crear_chunks(corpus, columna_texto='Texto_limpio',
#                                                tamano=150, solapamiento=20, umbral_minimo=100)
#
# DIFERENCIA con Karen: ella usa chunks solapados de 150 palabras porque trabaja
# con columnas de opinion (documentos largos). Los tweets tienen ~30 palabras
# -> cada tweet ES un chunk. Se mantiene la misma estructura de columnas para
# que el resto del pipeline (embeddings, BERTopic, etc.) funcione igual.
# ============================================================

cols_base = [
    'id_doc', COL_AUTOR, COL_AUTOR_NAME, COL_FECHA, COL_DATE,
    'anio', 'mes', 'anio_mes',
    COL_SENTIMIENTO, COL_POLARIDAD, COL_ENTIDAD,
    'Texto_limpio', 'n_palabras',
]
cols_base = [c for c in cols_base if c in corpus_limpio.columns]

chunks_df = corpus_limpio[cols_base].copy()

# Renombrar al esquema de Karen para compatibilidad downstream
chunks_df = chunks_df.rename(columns={
    COL_AUTOR:      'autor_doc',
    COL_AUTOR_NAME: 'nombre_autor_doc',
    COL_FECHA:      'fecha_doc',
    COL_DATE:       'date_doc',
    'Texto_limpio':  'texto_chunk',
    'n_palabras':    'n_palabras_chunk',
})

# Cada tweet = chunk 1 de 1
chunks_df['chunk_id']  = chunks_df['id_doc'].astype(str) + '_1'
chunks_df['num_chunk'] = 1

orden = [
    'id_doc', 'chunk_id', 'num_chunk',
    'autor_doc', 'nombre_autor_doc', 'fecha_doc', 'date_doc',
    'anio', 'mes', 'anio_mes',
    COL_SENTIMIENTO, COL_POLARIDAD, COL_ENTIDAD,
    'texto_chunk', 'n_palabras_chunk',
]
orden = [c for c in orden if c in chunks_df.columns]
chunks_df = chunks_df[orden].reset_index(drop=True)

print(f'[CHUNKS] Total chunks (= tweets) : {len(chunks_df):,}')
print(f'[CHUNKS] Columnas                : {list(chunks_df.columns)}')
print(f"[CHUNKS] Palabras/chunk  media: {chunks_df['n_palabras_chunk'].mean():.1f}  "
      f"mediana: {chunks_df['n_palabras_chunk'].median():.0f}  "
      f"max: {chunks_df['n_palabras_chunk'].max()}")
chunks_df.head(3)

In [ ]:
# ============================================================
# CELL 7 - GUARDAR CHUNKS
# Equivalente a: chunks_df.to_excel / to_parquet en Karen NB2
# ============================================================

chunks_parquet_path = DATA_PROCESSED / 'chunks.parquet'
chunks_excel_path   = DATA_PROCESSED / 'chunks.xlsx'

chunks_df.to_parquet(chunks_parquet_path, index=False, engine='pyarrow')
print(f'[GUARDADO] chunks.parquet : {len(chunks_df):,} filas')

chunks_df.to_excel(chunks_excel_path, index=False, engine='openpyxl')
print(f'[GUARDADO] chunks.xlsx    : {len(chunks_df):,} filas')
print('Chunks exportados a Excel y Parquet.')

import matplotlib.pyplot as plt
plt.hist(chunks_df['n_palabras_chunk'], bins=30, edgecolor='black')
plt.xlabel('Numero de palabras por tweet/chunk')
plt.ylabel('Frecuencia')
plt.title('Distribucion de palabras por chunk')
plt.show()
chunks_df['n_palabras_chunk'].describe()

In [ ]:
# ============================================================
# CELL 8 - VERIFICACION FINAL
# ============================================================
import os

print('=== RESUMEN NOTEBOOK 01 ===')
print(f'  Tweets tras limpieza  : {len(corpus_limpio):,}')
print(f'  Rango de fechas       : {corpus_limpio[COL_FECHA].min().date()} -> {corpus_limpio[COL_FECHA].max().date()}')
print(f'  Autores unicos        : {corpus_limpio[COL_AUTOR].nunique():,}')
print(f"  Media palabras/tweet  : {corpus_limpio['n_palabras'].mean():.1f}")
print()
print('  Archivos generados:')
for f in sorted(DATA_PROCESSED.iterdir()):
    size_kb = os.path.getsize(f) / 1024
    print(f'    {f.name:40s}  {size_kb:>8.1f} KB')
print()
print('Notebook 01 completado.')
print('Siguiente -> 02_estadisticos_corpus.ipynb')